## Data Cleaning Markdown
- Collect data from Hugging Face
- Subset for variables of interest
- Plot the variables of interest in different ways
- Inspect the data
- Check for NAs
- Clean

### PHASE 1: Data Cleaning

In [11]:
# Setup
from datasets import load_dataset
import numpy as np
import pandas as pd
import re
import os

In [6]:
# Load data set for only the food products
ds = load_dataset("openfoodfacts/product-database", split="food")

Generating food split: 4398832 examples [02:42, 27120.85 examples/s]
Generating beauty split: 57986 examples [00:01, 36662.23 examples/s]


In [7]:
# columns of interest
columns_of_interest = [
    "product_name",
    "brands",
    "nova_group",
    "nutriscore_score",
    "nutriscore_grade",
    "lang",
    "ingredients_original_tags",
    "code"
]

# convert to df
df = ds.select_columns(columns_of_interest).to_pandas()

In [12]:
# make data directory 
os.makedirs("data", exist_ok=True)

# save to csv for future use
df.to_csv("data/raw_data.csv", index=False)

In [13]:
# Inspect first few rows of the dataset
df.head()

,product_name,brands,nova_group,nutriscore_score,nutriscore_grade,lang,ingredients_original_tags,code
0,"[{'lang': 'main', 'text': 'Véritable pâte à ta...",Bovetti,NaN,25.0,e,fr,None,0000101209159
1,"[{'lang': 'main', 'text': 'Chamomile Herbal Te...",Lagg's,1.0,NaN,unknown,en,[en:camomile-flower],0000105000011
2,"[{'lang': 'main', 'text': 'Lagg's, herbal tea,...",Lagg's,1.0,NaN,unknown,en,[en:peppermint],0000105000042
3,"[{'lang': 'main', 'text': 'Linden Flowers Tea'...",Lagg's,NaN,NaN,unknown,en,[en:linden-flowers],0000105000059
4,"[{'lang': 'main', 'text': 'Herbal Tea, Hibiscu...",Lagg's,NaN,NaN,unknown,en,[en:roselle-flower],0000105000073


In [14]:
# See rows and columns
df.shape

(4398832, 8)

In [15]:
# filter for only rows where lang is 'en'
df = df[df["lang"] == "en"]

df.shape

(1827474, 8)

In [16]:
# Build a function to clean columns removing non-alphabetical characters
def lower_remove_nonalpha(name: str) -> str:
    if not isinstance(name, str):
        return name  # or return "" / raise — your call
    return re.sub(r'[^a-z]', '', name.lower())

df["product_name_clean"] = df["product_name"].apply(lower_remove_nonalpha)

df.shape

(1827474, 9)

In [17]:
# concatenate brand and product name to create a unique product ID - if brand is na then just use product name
df["product_ID"] = np.where(
    df["brands"].isna(),
    df["product_name_clean"].astype(str),
    df["brands"] + "_" + df["product_name_clean"].astype(str)
)

# make sure no products duplicated by "code" (barcode) - also unique product ID
if df["code"].nunique() != len(df):
    print("Warning: code column has duplicates.")

# for code duplicates keep the instance with the most data (least missing values)
df["missing_values"] = df.isna().sum(axis=1)
df = df.sort_values("missing_values").drop_duplicates(subset="code", keep="first").drop(columns="missing_values")

# double check for duplicates after dropping
if df["code"].nunique() != len(df):
    print("Warning: code column has duplicates.")
else:
    print("No duplicates in code column.")

No duplicates in code column.


In [18]:
df.shape

(1827468, 10)

### PHASE 2: Data Inspection & Plotting

In [19]:
# count na in each column
na_count = df.isna().sum()

# get percentage of missing values in each column
na_percent = df.isna().mean()*100

# combine into a dataframe
na_df = pd.DataFrame({"na_count": na_count, "na_percent": na_percent})
print(na_df)

                           na_count  na_percent
product_name                      0    0.000000
brands                       622872   34.083880
nova_group                  1302153   71.254490
nutriscore_score            1315441   71.981616
nutriscore_grade              37340    2.043264
lang                              0    0.000000
ingredients_original_tags   1203861   65.875900
code                              0    0.000000
product_name_clean                0    0.000000
product_ID                        0    0.000000


OBS: We have quite a lot of data without a nova group or a numerical nutriscore score. But many more do have a nutrigrade. So we should probably use the grade instead of the score - I will continue by using the nutrigrade.

Quite a lot of the data also does not have ingredients original tags - perhaps we need to check how much we have and if we use a different column from the OG data.

#### Create filtered df where there are ingredients and nutriscores

In [20]:
# filter for rows with a nutriscore_grade and ingredients_original_tags
df_filtered = df[df["nutriscore_grade"].notna() & df["ingredients_original_tags"].notna()]

df_filtered.shape

(623594, 10)

#### Create df with novas

In [21]:
# create df where nova is present
with_nova = df_filtered[df_filtered["nova_group"].notna()]

# save to csv for modelling
with_nova.to_csv("data/with_nova.csv", index=False)

with_nova.shape

(516374, 10)

In [22]:
# see how many products fall into each nova score for the with nova df
print(with_nova["nova_group"].value_counts())

nova_group
4.0    368966
3.0     87879
1.0     51227
2.0      8302
Name: count, dtype: int64


In [23]:
# see how many products fall into each nutriscore grade for the missing nova df
print(with_nova["nutriscore_grade"].value_counts())

nutriscore_grade
e                 121744
unknown           115706
d                  94465
c                  76251
a                  57771
b                  38137
not-applicable     12300
Name: count, dtype: int64


In [26]:
# exclude all rows with nutriscore_grade == "unknown" and "not-applicable"
with_nova_filtered = with_nova[~with_nova['nutriscore_grade'].isin(['unknown', 'not-applicable'])]

#Checking if they're gone
print(with_nova_filtered["nutriscore_grade"].value_counts())

nutriscore_grade
e    121744
d     94465
c     76251
a     57771
b     38137
Name: count, dtype: int64


#### Create df without novas

In [ ]:
# see how many rows of the filtered dataset have a nova group
missing_nova = df_filtered[df_filtered["nova_group"].isna()]

# save to csv for Nova imputation
missing_nova.to_csv("data/missing_nova.csv", index=False)

missing_nova.shape

(107126, 10)

In [ ]:
# percentage of df_filtered that doesn't have a nova
percent_missing_nova = (missing_nova.shape[0] / df_filtered.shape[0]) * 100
print(f"Percentage of df_filtered without nova_group: {percent_missing_nova:.2f}%")

Percentage of df_filtered without nova_group: 17.25%


In [ ]:
# see how many products fall into each nutriscore grade for the missing nova df
print(missing_nova["nutriscore_grade"].value_counts())

Issues to be addressed?
- For the nutriscore grade there is some called unknown and not-applicable -> need to see what we want to do with these? Filter out? or see what the use cases are?
    - not-applicable can be caused by various reasons, mainly:
        
        Alcoholic Beverages: Typically excluded from Nutri-Score calculation.
        
        Single-Ingredient Products: Foods like coffee, tea, herbal infusions, salt, vinegar, and yeast do not receive a grade because they are generally used in small quantities and contain negligible nutrients per serving. 
        
        Unprocessed Products: Raw products like fresh meat, fruit, or vegetables that are sold without additional ingredients often lack a label.
        
        Voluntary Participation: While adopted in several EU countries, Nutri-Score is voluntary, not mandatory. Companies may choose not to use it, leaving products unlabeled.
        
        Specialized Foods: Foods intended for particular nutritional uses (e.g., infant formula) are often exempt.
        
        Missing Information: Products with incomplete nutritional data in the database cannot be graded.

    MEANING: We should delete these, as they both may represent ultra-processed foods, very raw products, missing data AND products where it's simply not applicable. 
- There are varying amount in each nova group (when we split the data again it should be equal between all groups in each df? Amount of data could be seen as a weight?)
    - This will automatically be sorted later on within the stratified data splits. It's okay for now:)